# Customer Churn Prediction System
## Phase 4: Data Preprocessing

This notebook performs all data preprocessing steps required before model building.
The pipeline covers Missing Value Treatment, Categorical Encoding, Data Splitting,
and Feature Scaling - in the correct order to prevent data leakage.

Important: Feature Scaling is performed AFTER Data Splitting.
The scaler is fit ONLY on training data, then applied to validation and test sets.
This ensures no statistical information from test/validation data leaks into training.

## Setup - Import Libraries and Load Dataset

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# Load original raw dataset
df = pd.read_csv('../data/telco_customer_churn.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Dataset shape: (7043, 21)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


**Note:** The raw dataset is loaded fresh here with no changes carried over from the EDA notebook. All preprocessing steps are applied in this notebook only, keeping each phase cleanly separated.

---
## Activity 1 - Missing Value Treatment

Missing value treatment ensures the dataset has no gaps before model training.
Four methods are demonstrated below. The appropriate method is then applied to this dataset.

### Step 1 - Detect Missing Values

In [2]:
print("=== Formal Null Values (NaN) ===")
print(df.isnull().sum())

print("\n=== Blank Whitespace Entries ===")
print((df == " ").sum())

print("\n=== TotalCharges blank count ===")
print((df["TotalCharges"] == " ").sum())

=== Formal Null Values (NaN) ===
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

=== Blank Whitespace Entries ===
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        1

**Observation:** No formal NaN values exist. However, TotalCharges contains 11 blank whitespace entries - a known data quality issue identified during Phase 2. These blanks must be converted to NaN before any imputation method can be applied.

### Step 2 - Convert TotalCharges to Numeric
Converting the column from string/object type to float so imputation methods can be applied.

In [3]:
# Convert TotalCharges from string to float
# Blank strings become NaN automatically via errors='coerce'
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("=== TotalCharges dtype after conversion ===")
print(df['TotalCharges'].dtype)

print("\n=== NaN count in TotalCharges after conversion ===")
print(df['TotalCharges'].isnull().sum())

print("\n=== Sample of affected rows ===")
print(df[df['TotalCharges'].isnull()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

=== TotalCharges dtype after conversion ===
float64

=== NaN count in TotalCharges after conversion ===
11

=== Sample of affected rows ===
      customerID  tenure  MonthlyCharges  TotalCharges
488   4472-LVYGI       0           52.55           NaN
753   3115-CZMZD       0           20.25           NaN
936   5709-LVOEQ       0           80.85           NaN
1082  4367-NUYAO       0           25.75           NaN
1340  1371-DWPAZ       0           56.05           NaN
3331  7644-OMVMY       0           19.85           NaN
3826  3213-VVOLG       0           25.35           NaN
4380  2520-SGTTA       0           20.00           NaN
5218  2923-ARZLG       0           19.70           NaN
6670  4075-WKNIU       0           73.35           NaN
6754  2775-SEFEE       0           61.90           NaN


**Observation:** After conversion, TotalCharges is now float64 and the 11 blank entries are correctly showing as NaN. All 11 affected customers have 0 months of tenure - they were never billed, which explains the missing values.

### Step 3 - Missing Value Treatment Methods

Four standard methods are demonstrated below.

#### Method 1 - Mean Imputation
Replaces missing values with the column mean. Best for normally distributed numerical data with no significant outliers.

In [4]:
df_mean = df.copy()
mean_value = df_mean['TotalCharges'].mean()
df_mean['TotalCharges'] = df_mean['TotalCharges'].fillna(mean_value)

print(f"Mean value used for imputation: {mean_value:.2f}")
print(f"Remaining NaN in TotalCharges:  {df_mean['TotalCharges'].isnull().sum()}")

Mean value used for imputation: 2283.30
Remaining NaN in TotalCharges:  0


#### Method 2 - Median Imputation
Replaces missing values with the column median.
Best for skewed data or data with outliers - more robust than mean imputation.

In [5]:
df_median = df.copy()
median_value = df_median['TotalCharges'].median()
df_median['TotalCharges'] = df_median['TotalCharges'].fillna(median_value)

print(f"Median value used for imputation: {median_value:.2f}")
print(f"Remaining NaN in TotalCharges:   {df_median['TotalCharges'].isnull().sum()}")

Median value used for imputation: 1397.47
Remaining NaN in TotalCharges:   0


#### Method 3 - Mode Imputation
Replaces missing values with the most frequently occurring value.
Best for categorical columns or discrete numerical columns.

In [6]:
df_mode = df.copy()
mode_value = df_mode['TotalCharges'].mode()[0]
df_mode['TotalCharges'] = df_mode['TotalCharges'].fillna(mode_value)

print(f"Mode value used for imputation: {mode_value:.2f}")
print(f"Remaining NaN in TotalCharges:  {df_mode['TotalCharges'].isnull().sum()}")

Mode value used for imputation: 20.20
Remaining NaN in TotalCharges:  0


#### Method 4 - Record Removal
Removes all rows containing missing values.
Appropriate only when missing rows are very few relative to the dataset size.

In [7]:
df_removed = df.copy()
before = len(df_removed)
df_removed = df_removed.dropna(subset=['TotalCharges'])
after = len(df_removed)

print(f"Rows before removal: {before}")
print(f"Rows after removal:  {after}")
print(f"Rows removed:        {before - after}")

Rows before removal: 7043
Rows after removal:  7032
Rows removed:        11


### Step 4 - Apply Chosen Method to Main Dataset

In [8]:
# METHOD CHOSEN: Median Imputation
# Reason: TotalCharges is right-skewed (confirmed in EDA histograms).
# Median is more robust than mean for skewed distributions and
# is unaffected by high-value outliers present in TotalCharges.

df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

print("=== Chosen Method: Median Imputation ===")
print(f"Median value applied:              {df['TotalCharges'].median():.2f}")
print(f"Remaining NaN in TotalCharges:     {df['TotalCharges'].isnull().sum()}")
print(f"Dataset shape after treatment:     {df.shape}")

=== Chosen Method: Median Imputation ===
Median value applied:              1397.47
Remaining NaN in TotalCharges:     0
Dataset shape after treatment:     (7043, 21)


**Observation:** Median Imputation was chosen because TotalCharges is right-skewed as confirmed
in the EDA histogram. The median is a more robust central measure than the mean for skewed data.
Record Removal was not chosen because losing 11 rows is unnecessary when imputation is available.
All 11 missing values have been successfully filled.

---
## Activity 2 - Categorical Encoding

Machine learning models require all features to be numerical.
Categorical columns must be converted to numbers before training.
Two encoding methods are applied depending on the nature of each column.

### Step 1 - Drop customerID
customerID is a row identifier with no predictive value - it must be removed.

In [10]:
df.drop('customerID', axis=1, inplace=True)

print(f"customerID dropped.")
print(f"Dataset shape: {df.shape}")

customerID dropped.
Dataset shape: (7043, 20)


### Step 2 - Encode Target Variable

In [11]:
# Encode target: Yes = 1, No = 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("=== Target Variable Encoding ===")
print(df['Churn'].value_counts())
print(f"\nChurn dtype: {df['Churn'].dtype}")

=== Target Variable Encoding ===
Churn
0    5174
1    1869
Name: count, dtype: int64

Churn dtype: int64


### Step 3 - Method 1: Label Encoding
Label Encoding converts each category to an integer (0 or 1).
Applied to binary Yes/No columns where only 2 categories exist.

In [22]:
le = LabelEncoder()

# Binary columns - only 2 categories (Yes/No or Male/Female)
binary_cols = [
    'gender', 'Partner', 'Dependents',
    'PhoneService', 'PaperlessBilling'
]

label_encoded_cols = []

for col in binary_cols:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])
        label_encoded_cols.append(col)

print("=== Label Encoded Columns ===")
print(label_encoded_cols)
print("\n=== Sample after Label Encoding ===")
print(df[binary_cols].head())

=== Label Encoded Columns ===
[]

=== Sample after Label Encoding ===
   gender  Partner  Dependents  PhoneService  PaperlessBilling
0       0        1           0             0                 1
1       1        0           0             1                 0
2       1        0           0             1                 1
3       1        0           0             0                 0
4       0        0           0             1                 1


**Observation:** Label Encoding is applied to all binary categorical columns. Each Yes/No or
Male/Female column is now represented as 1/0. SeniorCitizen was already numeric (0/1) so
it does not need encoding and is excluded from this step.

### Step 4 - Method 2: One-Hot Encoding
One-Hot Encoding creates a separate binary column for each category value.
Applied to columns with 3 or more categories to avoid implying any ordinal relationship.
drop_first=True removes one dummy column per feature to avoid the dummy variable trap.

In [13]:
# Multi-class columns — 3 or more categories
ohe_cols = [
    'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod'
]

df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

print("=== Dataset shape after One-Hot Encoding ===")
print(df.shape)

print("\n=== All columns after encoding ===")
print(list(df.columns))

=== Dataset shape after One-Hot Encoding ===
(7043, 31)

=== All columns after encoding ===
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


**Observation:** One-Hot Encoding expanded the dataset from 20 columns to 31 columns
(30 input features + 1 target column Churn). drop_first=True dropped exactly 1 dummy
column per categorical feature to avoid multicollinearity. All features are now fully
numerical and ready for splitting and scaling.

---
## Activity 3 - Data Splitting

Data Splitting is performed BEFORE Feature Scaling.
This is a critical ML best practice - fitting the scaler on the full dataset before
splitting would cause data leakage: the scaler would learn the mean and standard deviation
of the test set, giving the model indirect access to test data during training.

The correct order is:
1. Split first → 2. Fit scaler on training data only → 3. Transform all three sets

Split: 70% Training / 15% Validation / 15% Testing (as per project instructions)
stratify=y is used to preserve the ~73.5% / 26.5% class ratio across all three sets.

In [14]:
# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"\nClass distribution:")
print(y.value_counts())

Features shape: (7043, 30)
Target shape:   (7043,)

Class distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [15]:
# Step 1 — Split into 70% train and 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Step 2 — Split the 30% temp equally into 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("=== Data Split Summary ===")
print(f"Total samples:   {len(X)}")
print(f"Training set:    {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation set:  {len(X_val)}  ({len(X_val)/len(X)*100:.1f}%)")
print(f"Testing set:     {len(X_test)}  ({len(X_test)/len(X)*100:.1f}%)")

=== Data Split Summary ===
Total samples:   7043
Training set:    4930 (70.0%)
Validation set:  1056  (15.0%)
Testing set:     1057  (15.0%)


In [16]:
# Verify class balance is preserved in each split
print("=== Churn Distribution — Training Set ===")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\n=== Churn Distribution — Validation Set ===")
print(y_val.value_counts(normalize=True).mul(100).round(2))

print("\n=== Churn Distribution — Testing Set ===")
print(y_test.value_counts(normalize=True).mul(100).round(2))

=== Churn Distribution — Training Set ===
Churn
0    73.47
1    26.53
Name: proportion, dtype: float64

=== Churn Distribution — Validation Set ===
Churn
0    73.48
1    26.52
Name: proportion, dtype: float64

=== Churn Distribution — Testing Set ===
Churn
0    73.42
1    26.58
Name: proportion, dtype: float64


**Observation:** The dataset was split into 70% training (4,930 samples), 15% validation
(1,056 samples), and 15% testing (1,057 samples). stratify=y ensures the ~73.5% / 26.5%
churn class ratio is preserved across all three sets, preventing any set from being
accidentally dominated by one class.

---
## Activity 4 - Feature Scaling

Scaling is performed AFTER splitting - this is the correct production-grade order.

The scaler is fit ONLY on X_train, then used to transform X_val and X_test.
This means the scaler learns only the training data's mean and standard deviation.
The validation and test sets are transformed using those training statistics -
exactly as would happen with completely new, unseen data in production.

Both methods are demonstrated on copies before applying the chosen method.

### Step 1 - Identify Columns to Scale

In [17]:
scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

print("=== Before Scaling — Training Set Statistics ===")
print(X_train[scale_cols].describe().round(2))

=== Before Scaling — Training Set Statistics ===
        tenure  MonthlyCharges  TotalCharges
count  4930.00         4930.00       4930.00
mean     32.47           64.95       2303.78
std      24.65           30.24       2291.18
min       0.00           18.40         18.85
25%       9.00           35.50        401.20
50%      29.00           70.57       1390.72
75%      56.00           90.05       3857.01
max      72.00          118.75       8684.80


### Step 2 - Method 1: StandardScaler (demonstrated on copies)
Transforms data to have mean = 0 and standard deviation = 1.
Fit on training data only, then transform all three sets.

In [18]:
X_train_std = X_train.copy()
X_val_std   = X_val.copy()
X_test_std  = X_test.copy()

scaler_std = StandardScaler()

# FIT on training only — TRANSFORM all three
X_train_std[scale_cols] = scaler_std.fit_transform(X_train[scale_cols])
X_val_std[scale_cols]   = scaler_std.transform(X_val[scale_cols])
X_test_std[scale_cols]  = scaler_std.transform(X_test[scale_cols])

print("=== After StandardScaler — Training Set ===")
print(X_train_std[scale_cols].describe().round(4))

=== After StandardScaler — Training Set ===
          tenure  MonthlyCharges  TotalCharges
count  4930.0000       4930.0000     4930.0000
mean      0.0000          0.0000       -0.0000
std       1.0001          1.0001        1.0001
min      -1.3176         -1.5393       -0.9974
25%      -0.9524         -0.9739       -0.8305
50%      -0.1409          0.1860       -0.3986
75%       0.9548          0.8300        0.6780
max       1.6040          1.7791        2.7853


### Step 3 - Method 2: MinMaxScaler (demonstrated on copies)
Transforms data to a fixed range of [0, 1].
Fit on training data only, then transform all three sets.

In [19]:
X_train_mm = X_train.copy()
X_val_mm   = X_val.copy()
X_test_mm  = X_test.copy()

scaler_mm = MinMaxScaler()

# FIT on training only — TRANSFORM all three
X_train_mm[scale_cols] = scaler_mm.fit_transform(X_train[scale_cols])
X_val_mm[scale_cols]   = scaler_mm.transform(X_val[scale_cols])
X_test_mm[scale_cols]  = scaler_mm.transform(X_test[scale_cols])

print("=== After MinMaxScaler — Training Set ===")
print(X_train_mm[scale_cols].describe().round(4))

=== After MinMaxScaler — Training Set ===
          tenure  MonthlyCharges  TotalCharges
count  4930.0000       4930.0000     4930.0000
mean      0.4510          0.4639        0.2637
std       0.3423          0.3014        0.2644
min       0.0000          0.0000        0.0000
25%       0.1250          0.1704        0.0441
50%       0.4028          0.5199        0.1583
75%       0.7778          0.7140        0.4429
max       1.0000          1.0000        1.0000


### Step 4 - Apply Chosen Scaler to Final Arrays

In [20]:
# METHOD CHOSEN: StandardScaler
# Reason: tenure and MonthlyCharges are approximately normally distributed
# as confirmed in EDA. StandardScaler is the standard choice for
# Logistic Regression, SVM, and most classification algorithms used in Phase 6.

# Create explicit copies to avoid pandas SettingWithCopyWarning
X_train_scaled = X_train.copy()
X_val_scaled   = X_val.copy()
X_test_scaled  = X_test.copy()

scaler = StandardScaler()

# ✅ CORRECT ORDER — fit ONLY on training, transform all three
X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val_scaled[scale_cols]   = scaler.transform(X_val[scale_cols])
X_test_scaled[scale_cols]  = scaler.transform(X_test[scale_cols])

print("=== Chosen Method: StandardScaler ===")
print("Fit on: Training set only")
print("Transformed: Training, Validation, and Testing sets")
print(f"\nTraining set mean after scaling:  {X_train_scaled[scale_cols].mean().round(4).to_dict()}")
print(f"Training set std after scaling:   {X_train_scaled[scale_cols].std().round(4).to_dict()}")

=== Chosen Method: StandardScaler ===
Fit on: Training set only
Transformed: Training, Validation, and Testing sets

Training set mean after scaling:  {'tenure': 0.0, 'MonthlyCharges': 0.0, 'TotalCharges': -0.0}
Training set std after scaling:   {'tenure': 1.0001, 'MonthlyCharges': 1.0001, 'TotalCharges': 1.0001}


**Observation:** StandardScaler was chosen because tenure and MonthlyCharges follow approximately
normal distributions as confirmed in the EDA histograms. The scaler was fit exclusively on
X_train - the validation and test sets were only transformed using the training statistics.
This is the correct production-grade approach and prevents data leakage completely.

---
## Save Cleaned Dataset and Splits - Deliverable 4

In [21]:
# Save the full preprocessed dataset
df.to_csv('../data/cleaned_dataset.csv', index=False)

# Save the scaled feature splits with their matching target columns
X_train_scaled.assign(Churn=y_train.values).to_csv('../data/X_train.csv', index=False)
X_val_scaled.assign(Churn=y_val.values).to_csv('../data/X_val.csv', index=False)
X_test_scaled.assign(Churn=y_test.values).to_csv('../data/X_test.csv', index=False)

print("=== Files Saved ===")
print("cleaned_dataset.csv  — full preprocessed dataset (7,043 × 31)")
print("X_train.csv          — scaled training set with Churn (70%)")
print("X_val.csv            — scaled validation set with Churn (15%)")
print("X_test.csv           — scaled testing set with Churn (15%)")

=== Files Saved ===
cleaned_dataset.csv  — full preprocessed dataset (7,043 × 31)
X_train.csv          — scaled training set with Churn (70%)
X_val.csv            — scaled validation set with Churn (15%)
X_test.csv           — scaled testing set with Churn (15%)


**Observation:** The cleaned dataset and all three scaled split files have been saved to the
data/ folder. These files will be loaded directly in Phase 6 (Model Building) without any
further preprocessing needed.

---
## Preprocessing Pipeline Summary — Deliverable 4

| Step | Activity | Method Applied | Reason |
|------|----------|----------------|--------|
| 1 | Missing Value Detection | isnull() + blank string check | TotalCharges had 11 hidden blank entries |
| 2 | Type Conversion | pd.to_numeric(errors='coerce') | TotalCharges was stored as string/object |
| 3 | Missing Value Treatment | Median Imputation on TotalCharges | Right-skewed distribution — median more robust than mean |
| 4 | Drop Identifier | Removed customerID | No predictive value — row identifier only |
| 5 | Target Encoding | Churn: Yes=1, No=0 | Required for binary classification |
| 6 | Label Encoding | gender, Partner, Dependents, PhoneService, PaperlessBilling | Binary columns — 2 categories only |
| 7 | One-Hot Encoding | MultipleLines, InternetService, Contract, PaymentMethod + others | 3+ categories — avoids false ordinal ranking |
| 8 | Data Splitting | 70% Train / 15% Val / 15% Test with stratify=y | Split BEFORE scaling to prevent data leakage |
| 9 | Feature Scaling | StandardScaler fit on X_train only, transform all three sets | Prevents data leakage — test set statistics never seen by scaler |

### Final Dataset Statistics
- Original shape:  7,043 rows × 21 columns
- Cleaned shape:   7,043 rows × 31 columns (30 input features + 1 target Churn)
- Missing values:  0 remaining
- All features:    fully numerical
- Training set:    4,930 samples (70%)
- Validation set:  1,056 samples (15%)
- Testing set:     1,057 samples (15%)
- Files saved:     cleaned_dataset.csv + X_train.csv + X_val.csv + X_test.csv